# 📈 GIADA Task 5 — scaling fisico e numerico
Tre capacità physical-τ e tredici baseline numeriche, con budget, dati e benchmark GPU allineati.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_5'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 5 richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,PrimitiveScalingConfig,verified_task4_root,prepare_scaling_data,train_and_freeze_scaling,evaluate_frozen_scaling
from src.giada_teacher.primitive_scaling import EXPECTED_TASK4_ARCHIVE_SHA256,EXPECTED_TASK4_REPORT_SHA256
prereg=json.loads((GIADA_REPO/'experiments/teacher_primitive_scaling_preregistration_v1.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'preregistration':prereg})


In [ ]:
def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()
INPUT_ROOT=Path('/kaggle/input'); override=os.environ.get('GIADA_TASK4_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
    candidates += list(INPUT_ROOT.rglob('giada_paired_primitive_matrix.zip'))
    candidates += list(INPUT_ROOT.rglob('archive.zip'))
    candidates += [p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'frozen_primitive_checkpoints.pt').is_file()]
def exact(path):
    try: return file_sha(path)==EXPECTED_TASK4_ARCHIVE_SHA256 if path.is_file() else file_sha(path/'final_report.json')==EXPECTED_TASK4_REPORT_SHA256
    except Exception: return False
TASK4_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact(p)),None)
assert TASK4_SOURCE is not None,'Aggiungi agli Input Kaggle l’artefatto esatto giada_paired_primitive_matrix.zip.'
TASK4_ROOT=verified_task4_root(TASK4_SOURCE,Path('/kaggle/working/.task4_verified'))
print({'task4_source':str(TASK4_SOURCE),'authorization_verified':True})


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_primitive_scaling_laws')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
config=PrimitiveScalingConfig(); bundle=prepare_scaling_data(formula,config)
display({'widths':config.widths,'seeds':config.seeds,'budgets':config.checkpoints,'lut_points':config.lut_points,'chebyshev_degrees':config.chebyshev_degrees,'sealed_not_materialized_yet':True})


## 🚀 Training e freeze
Tre larghezze × tre seed avanzano nello stesso ciclo. Una riga compatta a ogni checkpoint mostra percentuale, ETA e score per larghezza.

In [ ]:
training=train_and_freeze_scaling(bundle,OUTPUT_DIR,config,code_revision=REVISION)
display({'valid':training['valid'],'selection':training['selection'],'same_minibatches_across_widths':training['same_minibatches_across_widths']})
assert training['valid'] and not training['task4_sealed_accessed']


## 🔒 Sealed Task 5 e benchmark sulla stessa GPU
Il nuovo sealed viene aperto solo dopo il freeze. Questa cella confronta accuratezza, risoluzione, memoria e throughput.

In [ ]:
final=evaluate_frozen_scaling(bundle,OUTPUT_DIR,config)
display({'valid':final['valid'],'sealed':final['sealed_contract'],'decision':final['decision'],'physical':{k:{'score':v['mean_score'],'parameters':v['parameter_count_per_seed'],'gpu':v['gpu']} for k,v in final['physical'].items()},'numerical':{k:{'score':v['score'],'bytes':v['table_bytes'],'gpu':v['gpu']} for k,v in final['numerical'].items()}})
assert final['valid'] and not final['selection_used_sealed'] and not final['task4_sealed_accessed']


## 📦 Scarica lo ZIP con il metodo Blob/base64

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_primitive_scaling_laws','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
